# Dependency Test Notebook

Minimal test to validate Phase 1 dependencies before running full collection.

**What this does:**
- Tests specific dependency versions
- Tests basic imports
- Attempts model load (without forward pass)
- Reports which versions work
- Then you update requirements.txt and re-run to confirm

**Runtime:** ~5-10 minutes

In [1]:
# Clone repo (use integration branch with Phase 1 code)
!rm -rf /content/moe-expert-prefetching
!cd /content && git clone --branch integration/phase-1-validation https://github.com/scerruti/moe-expert-prefetching.git
%cd /content/moe-expert-prefetching/phase_1/scripts

Cloning into 'moe-expert-prefetching'...
remote: Enumerating objects: 425, done.
remote: Counting objects: 100% (287/287), done.
remote: Compressing objects: 100% (201/201), done.
remote: Total 425 (delta 149), reused 172 (delta 85), pack-reused 138 (from 1)
Receiving objects: 100% (425/425), 1.36 MiB | 38.68 MiB/s, done.
Resolving deltas: 100% (194/194), done.
/content/moe-expert-prefetching/phase_1/scripts


In [ ]:
# Test specific versions (not from requirements.txt yet)
# Using versions that actually exist on PyPI and work
print("Testing versions:")
print("  torch==2.15.0")
print("  torchvision==0.29.2")
print("  transformers==4.45.0")
print("  datasets==5.0.1")
print("  pyarrow==25.0.1\n")

!pip install -q torch==2.15.0 torchvision==0.29.2 transformers==4.45.0 datasets==5.0.1 pyarrow==25.0.1 tqdm accelerate bitsandbytes
print("\n✅ Test versions installed")

In [3]:
# Test 1: Basic imports
print("Test 1: Basic imports...")
try:
    import torch
    import transformers
    import datasets
    import pyarrow
    print(f"✅ torch {torch.__version__}")
    print(f"✅ transformers {transformers.__version__}")
    print(f"✅ datasets {datasets.__version__}")
    print(f"✅ pyarrow {pyarrow.__version__}")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    raise

Test 1: Basic imports...
✅ torch 2.15.0+cu132
✅ transformers 4.45.0
✅ datasets 5.0.1
✅ pyarrow 25.0.1


In [5]:
# Test 2: Module imports
print("\nTest 2: Phase 1 module imports...")
try:
    import sys
    sys.path.insert(0, '/content/moe-expert-prefetching/phase_1/scripts')
    
    from model_loader import ModelLoader
    from dataset_loaders import GSM8KLoader, MBPPLoader
    from tokenizer import TokenizationPipeline
    print("✅ ModelLoader imported")
    print("✅ Dataset loaders imported")
    print("✅ TokenizationPipeline imported")
except ImportError as e:
    print(f"❌ Module import failed: {e}")
    raise


Test 2: Phase 1 module imports...
✅ ModelLoader imported
✅ Dataset loaders imported
✅ TokenizationPipeline imported


In [6]:
# Test 3: Model loading
print("\nTest 3: Model loading (this takes ~2 min)...")
try:
    loader = ModelLoader("qwen1.5-moe-a2.7b")
    print(f"  Loading {loader.model_id}...")
    model, tokenizer, config = loader.load()
    print(f"✅ Model loaded successfully")
    print(f"  Device: {loader.device}")
    print(f"  Dtype: {loader.dtype}")
    print(f"  Experts: {config['num_experts']}, Top-K: {config['top_k']}")
except Exception as e:
    print(f"❌ Model loading failed: {e}")
    raise


Test 3: Model loading (this takes ~2 min)...
  Loading Qwen/Qwen1.5-MoE-A2.7B-Chat...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/248 [00:00<?, ?B/s]

✅ Model loaded successfully
  Device: cuda
  Dtype: torch.float16
  Experts: 60, Top-K: 4


In [7]:
# Test 4: GPU verification
print("\nTest 4: GPU availability...")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("✅ GPU ready for Phase 1 collection")
else:
    print("⚠️  No GPU available (Phase 1 will run very slowly)")


Test 4: GPU availability...
GPU Available: True
GPU Name: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB
✅ GPU ready for Phase 1 collection


In [ ]:
# Summary
print("\n" + "="*60)
print("✅ ALL DEPENDENCY TESTS PASSED")
print("="*60)
print("\nThese versions work:")
print("  torch==2.15.0")
print("  torchvision==0.29.2")
print("  transformers==4.45.0")
print("  datasets==5.0.1")
print("  pyarrow==25.0.1")
print("\nNext steps:")
print("1. Update requirements.txt with these versions if not already done")
print("2. Then run Phase1_Full_Collection_Colab.ipynb for Phase 1")